# 03 · PCAP → JSON for LLM prompts

Stage 2 data preparation. Uses `pyshark` (tshark) to turn each 5-second window into a JSON object with a general summary (packet count, protocols) and a **packet-by-packet text log** (timestamp, src → dst, protocol, ports, TCP flags, DNS query / HTTP URI / ARP who-has). Logs are capped at 50 packets per window to fit the LLM context. The last cell merges all per-capture JSON files into `dataset.json` (167 windows).

Note: `arp_ve_dns_saldirisi.pcapng` is truncated and could not be parsed by tshark, so it is not part of `dataset.json`.


In [10]:
!pip install nest-asyncio

In [11]:
!pip install pyshark

In [12]:
!apt-get update
!apt-get install tshark
!pip install pyshark

In [14]:
import nest_asyncio
nest_asyncio.apply()

import pyshark
import json
from collections import defaultdict

def process_pcap_detailed(pcap_file, output_json, label, time_window=5.0):
    cap = pyshark.FileCapture(pcap_file)

    # Zaman pencerelerini tutacağımız yapı
    windows = defaultdict(lambda: {
        "genel_ozet": {
            "paket_sayisi": 0,
            "protokoller": set(),
        },
        "gozlemlenen_davranis_oruntuleri": [] # Paket paket akışın tutulacağı liste
    })

    start_time = None

    for pkt in cap:
        try:
            if start_time is None:
                start_time = float(pkt.sniff_timestamp)

            current_time = float(pkt.sniff_timestamp)
            window_index = int((current_time - start_time) // time_window)

            w = windows[window_index]
            w["genel_ozet"]["paket_sayisi"] += 1
            w["genel_ozet"]["protokoller"].add(pkt.highest_layer)

            # Kaynak ve hedef IP'lerini al
            src_ip = pkt.ip.src if hasattr(pkt, 'ip') else getattr(pkt.eth, 'src', 'Bilinmeyen')
            dst_ip = pkt.ip.dst if hasattr(pkt, 'ip') else getattr(pkt.eth, 'dst', 'Bilinmeyen')

            # 1. Adım: Temel paket bilgisi
            paket_detayi = f"[{current_time - start_time:.2f}s] {src_ip} -> {dst_ip} | Protokol: {pkt.highest_layer}"

            # 2. Adım: Hocanın istediği içerik özetleri ve port bilgileri
            if hasattr(pkt, 'tcp'):
                paket_detayi += f" | Port: {pkt.tcp.srcport}->{pkt.tcp.dstport} | TCP Flags: {pkt.tcp.flags}"
            elif hasattr(pkt, 'udp'):
                paket_detayi += f" | Port: {pkt.udp.srcport}->{pkt.udp.dstport}"

            # Uygulama katmanı içerik özetleri
            if hasattr(pkt, 'dns') and hasattr(pkt.dns, 'qry_name'):
                paket_detayi += f" | DNS Sorgusu: {pkt.dns.qry_name}"
            elif hasattr(pkt, 'http') and hasattr(pkt.http, 'request_uri'):
                paket_detayi += f" | HTTP İstek: {pkt.http.request_uri}"
            elif hasattr(pkt, 'arp'):
                paket_detayi += f" | ARP: Kimde {getattr(pkt.arp, 'dst_proto_ipv4', '')}? Söyle {getattr(pkt.arp, 'src_proto_ipv4', '')}"

            w["gozlemlenen_davranis_oruntuleri"].append(paket_detayi)

        except AttributeError:
            continue

    cap.close()

    # JSON formatına uygun hale getirme
    dataset = []
    for idx, data in windows.items():
        if data["genel_ozet"]["paket_sayisi"] > 0:
            dataset.append({
                "zaman_araligi_sn": time_window,
                "etiket": label,
                "genel_ozet": {
                    "paket_sayisi": data["genel_ozet"]["paket_sayisi"],
                    "protokoller": list(data["genel_ozet"]["protokoller"]),
                },
                # LLM'in token sınırını aşmamak için bir penceredeki maksimum paket sayısına sınır koyabiliriz (örn: ilk 50 paket)
                "gozlemlenen_davranis_oruntuleri": data["gozlemlenen_davranis_oruntuleri"][:50]
            })

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, ensure_ascii=False, indent=4)

    print(f"{pcap_file} detaylı şekilde işlendi ve {output_json} dosyasına kaydedildi.")

# Örnek Kullanım:
#process_pcap_detailed("normal_trafik.pcapng", "normal.json", label="normal")
#process_pcap_detailed("normal_trafik2.pcapng", "normal2.json", label="normal")
#process_pcap_detailed("normal_trafik3.pcapng", "normal3.json", label="normal")
#process_pcap_detailed("normal_trafik4.pcapng", "normal4.json", label="normal")

#process_pcap_detailed("nmap_tarama.pcapng", "nmap_tarama.json", label="zararli")
#process_pcap_detailed("dhcp_poisoning_trafigi.pcapng", "dhcp_poisoning_trafigi.json", label="zararli")
#process_pcap_detailed("arp_ve_dns_saldirisi.pcapng", "arp_ve_dns_saldirisi.json", label="zararli")
process_pcap_detailed("arp_saldiri_trafigi.pcapng", "arp_saldiri_trafigi.json", label="zararli")


arp_saldiri_trafigi.pcapng detaylı şekilde işlendi ve arp_saldiri_trafigi.json dosyasına kaydedildi.


In [15]:
import json
import glob
import os

def birlestir_json_dosyalari(cikis_dosyasi="dataset.json"):
    tum_veriler = []

    # Klasördeki tüm .json uzantılı dosyaları bul
    json_dosyalari = glob.glob("*.json")

    # Eğer daha önceden oluşturulmuş bir dataset.json varsa, onu tekrar okumamak için listeden çıkar
    if cikis_dosyasi in json_dosyalari:
        json_dosyalari.remove(cikis_dosyasi)

    if not json_dosyalari:
        print("Birleştirilecek hiçbir JSON dosyası bulunamadı!")
        return

    print(f"Birleştirilecek dosyalar: {json_dosyalari}\n")

    # Her bir dosyayı sırayla oku ve ana listeye ekle
    for dosya in json_dosyalari:
        try:
            with open(dosya, 'r', encoding='utf-8') as f:
                veri = json.load(f)

                # Bizim oluşturduğumuz JSON'lar bir liste formundaydı, bu yüzden extend kullanıyoruz
                if isinstance(veri, list):
                    tum_veriler.extend(veri)
                    print(f"[+] {dosya} başarıyla eklendi. (İçinden {len(veri)} örnek alındı)")
                else:
                    print(f"[!] UYARI: {dosya} beklenen liste formatında değil, atlanıyor.")

        except Exception as e:
            print(f"[-] HATA: {dosya} okunurken bir sorun oluştu: {e}")

    # Topladığımız tüm verileri tek bir JSON dosyasına yazdır
    with open(cikis_dosyasi, 'w', encoding='utf-8') as f:
        json.dump(tum_veriler, f, ensure_ascii=False, indent=4)

    print(f"\nİşlem Tamamlandı! Toplam {len(tum_veriler)} ağ trafiği örneği '{cikis_dosyasi}' dosyasına kaydedildi.")

# Kodu çalıştır
birlestir_json_dosyalari()

Birleştirilecek dosyalar: ['normal2.json', 'normal.json', 'normal3.json', 'arp_saldiri_trafigi.json', 'dhcp_poisoning_trafigi.json', 'nmap_tarama.json', 'normal4.json']

[+] normal2.json başarıyla eklendi. (İçinden 42 örnek alındı)
[+] normal.json başarıyla eklendi. (İçinden 46 örnek alındı)
[+] normal3.json başarıyla eklendi. (İçinden 8 örnek alındı)
[+] arp_saldiri_trafigi.json başarıyla eklendi. (İçinden 37 örnek alındı)
[+] dhcp_poisoning_trafigi.json başarıyla eklendi. (İçinden 6 örnek alındı)
[+] nmap_tarama.json başarıyla eklendi. (İçinden 7 örnek alındı)
[+] normal4.json başarıyla eklendi. (İçinden 21 örnek alındı)

İşlem Tamamlandı! Toplam 167 ağ trafiği örneği 'dataset.json' dosyasına kaydedildi.
